# 02 · Silver — limpo, tipado, deduplicado

Tudo o que a Silver faz tem uma **regra explícita**. O que viola a regra não some: vai para a **quarentena**.

| Tabela | Regras |
|---|---|
| clientes | `cliente_id` int; `estado` em maiúsculas; e-mail em minúsculas, inválido vira `NULL`; datas em 3 formatos; fica a versão mais recente por `cliente_id` |
| produtos | `preco` e `custo` > 0, senão quarentena |
| pedidos | tipos corretos; deduplica por `pedido_id`; **cliente inexistente → quarentena**; `itens` explodem em `itens_pedido` |
| avaliacoes | `nota` entre 1 e 5, senão quarentena; deduplica por `avaliacao_id` |

A Silver é reconstruída a partir de **toda** a Bronze (todos os lotes), então é segura para reexecutar.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

Spark 3.5.3 | master: local[2] | storage: minio


## Antes da função: faça na mão
Na aula de Spark você explodiu a coluna `itens`. Agora vamos fazer isso **a partir da Bronze**, que tem uma diferença: ela guarda os pedidos duplicados.
Se explodirmos antes de deduplicar, cada item duplicado conta duas vezes e o faturamento fica inflado.

In [2]:
from pyspark.sql.window import Window

brz = spark.table("lake.bronze.pedidos")
print("Bronze:", brz.count(), "linhas |", brz.select("pedido_id").distinct().count(), "pedidos distintos")

# 1) deduplica por pedido_id, ficando com a versão mais recente
w = Window.partitionBy("pedido_id").orderBy(F.col("_ingest_ts").desc())
pedidos_unicos = brz.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
print("Duplicados descartados:", brz.count() - pedidos_unicos.count())

# 2) explode itens (posexplode guarda a posição do item no pedido)
itens = (pedidos_unicos
    .select("pedido_id", "cliente_id", "data_pedido", "status",
            F.posexplode("itens").alias("item_seq", "item"))
    .select("pedido_id", "cliente_id", "data_pedido", "status", "item_seq",
            F.col("item.produto_id").alias("produto_id"),
            F.col("item.qtd").alias("qtd"),
            F.col("item.preco_unit").alias("preco_unit"))
    .withColumn("valor_item", F.round(F.col("qtd") * F.col("preco_unit"), 2)))

print("pedidos:", pedidos_unicos.count(), "| itens:", itens.count())
itens.show(5, truncate=False)

Bronze: 30150 linhas | 30000 pedidos distintos
Duplicados descartados: 150
pedidos: 30000 | itens: 50105
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
|pedido_id|cliente_id|data_pedido        |status   |item_seq|produto_id|qtd|preco_unit|valor_item|
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
|100001   |1257      |2025-09-21T23:11:30|entregue |0       |200       |1  |595.8     |595.8     |
|100002   |1959      |2025-06-05T23:59:12|cancelado|0       |101       |1  |872.69    |872.69    |
|100002   |1959      |2025-06-05T23:59:12|cancelado|1       |82        |3  |220.7     |662.1     |
|100002   |1959      |2025-06-05T23:59:12|cancelado|2       |57        |1  |460.02    |460.02    |
|100002   |1959      |2025-06-05T23:59:12|cancelado|3       |142       |2  |577.76    |1155.52   |
+---------+----------+-------------------+---------+--------+----------+---+----------+----------+
only

### Checagens de qualidade nos itens
Mesmas três perguntas da aula (valores inválidos, chave duplicada, produto inexistente), mas agora quem falha **não some**: ganha um motivo e vai para a quarentena.

In [3]:
produtos = (spark.table("lake.bronze.produtos")
            .select(F.col("produto_id").cast("long").alias("produto_id"))
            .distinct().withColumn("_existe", F.lit(True)))

valor_ok = (F.col("qtd") > 0) & (F.col("preco_unit") > 0) & F.col("produto_id").isNotNull()

itens_f = (itens.join(produtos, "produto_id", "left")
    .withColumn("motivo",
        F.when(F.coalesce(~valor_ok, F.lit(True)), "valor_invalido")
         .when(F.col("_existe").isNull(), "produto_inexistente")))

itens_ok  = itens_f.filter("motivo IS NULL").drop("motivo", "_existe")
itens_rej = itens_f.filter("motivo IS NOT NULL")

print("itens válidos:", itens_ok.count(), "| rejeitados:", itens_rej.count())
itens_rej.groupBy("motivo").count().show()

# chave composta única (pedido_id, item_seq) nos válidos
dup = itens_ok.groupBy("pedido_id", "item_seq").count().filter("count > 1").count()
print("chaves duplicadas:", dup, "->", "OK" if dup == 0 else "ERRO")

itens válidos: 50105 | rejeitados: 0
+------+-----+
|motivo|count|
+------+-----+
+------+-----+

chaves duplicadas: 0 -> OK


In [4]:
# Salvar na Silver

itens_ok.writeTo("lake.silver.itens_pedido_aula").using("iceberg").createOrReplace()
spark.table("lake.silver.itens_pedido_aula").count()

50105

In [5]:
resumo = M.build_silver(spark)
resumo

{'clientes': 5000,
 'produtos': 197,
 'pedidos': 29960,
 'itens_pedido': 50030,
 'avaliacoes': 6464,
 'quarentena': 68}

## Reconciliação: tudo que entrou tem que estar em algum lugar
Regra de ouro: **entrada = saída + quarentena**. Se a conta não fecha, o pipeline está perdendo dado sem avisar.

In [6]:
n_bronze = spark.table("lake.bronze.pedidos").select("pedido_id").distinct().count()
n_silver = spark.table("lake.silver.pedidos").count()
n_quar   = spark.table("lake.silver.quarentena").filter("origem = 'pedidos_cliente_orfao'").count()
print(f"pedidos : {n_bronze} = {n_silver} (silver) + {n_quar} (quarentena) -> {n_bronze == n_silver + n_quar}")

n_mao    = spark.table("lake.silver.itens_pedido_aula").count()
n_funcao = spark.table("lake.silver.itens_pedido").count()
print(f"itens   : feito na mão = {n_mao} | feito pela função = {n_funcao}")

pedidos : 30000 = 29960 (silver) + 40 (quarentena) -> True
itens   : feito na mão = 50105 | feito pela função = 50030


## O que foi para a quarentena?

In [7]:
spark.table("lake.silver.quarentena").groupBy("origem").count().show(truncate=False)
spark.table("lake.silver.quarentena").show(5, truncate=False)

+------------------------+-----+
|origem                  |count|
+------------------------+-----+
|pedidos_cliente_orfao   |40   |
|produtos                |3    |
|avaliacoes_nota_invalida|25   |
+------------------------+-----+

+---------------------+-------------------------------------------------+-------------------------+
|origem               |registro                                         |_quarentena_ts           |
+---------------------+-------------------------------------------------+-------------------------+
|produtos             |{"produto_id":55,"preco":-418.73,"custo":328.22} |2026-10-24 21:43:09.84787|
|produtos             |{"produto_id":114,"preco":-462.01,"custo":345.79}|2026-10-24 21:43:09.84787|
|produtos             |{"produto_id":173,"preco":-308.2,"custo":231.93} |2026-10-24 21:43:09.84787|
|pedidos_cliente_orfao|{"pedido_id":103054,"cliente_id":999999}         |2026-10-24 21:43:09.84787|
|pedidos_cliente_orfao|{"pedido_id":104139,"cliente_id":999999}     

## Data Quality: medir, não só limpar
`DQ` aplica regras e devolve um relatório. Guardar o relatório numa tabela é o começo da **observabilidade**.

In [8]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python:", sys.executable)
print("SPARK_LOCAL_IP:", os.environ["SPARK_LOCAL_IP"])

Python: c:\Users\giova\AppData\Local\Programs\Python\Python310\python.exe
SPARK_LOCAL_IP: 127.0.0.1


In [9]:
# Executa as validações diretamente via Spark SQL (sem UDFs nem workers Python)
rel = spark.sql("""
WITH c_cli AS (
    SELECT 
        count(*) AS total,
        count(CASE WHEN cliente_id IS NULL THEN 1 END) AS nulos_id,
        count(*) - count(DISTINCT cliente_id) AS dup_id,
        count(CASE WHEN email IS NULL THEN 1 END) AS nulos_email
    FROM lake.silver.clientes
),
c_ped AS (
    SELECT 
        count(*) AS total,
        count(CASE WHEN pedido_id IS NULL THEN 1 END) AS nulos_id,
        count(*) - count(DISTINCT pedido_id) AS dup_id,
        count(CASE WHEN status NOT IN ('pago', 'enviado', 'entregue', 'cancelado', 'devolvido') THEN 1 END) AS inv_status
    FROM lake.silver.pedidos
),
c_av AS (
    SELECT 
        count(*) AS total,
        count(*) - count(DISTINCT avaliacao_id) AS dup_id,
        count(CASE WHEN nota < 1 OR nota > 5 THEN 1 END) AS inv_nota
    FROM lake.silver.avaliacoes
)
SELECT 'silver.clientes' AS tabela, 'not_null' AS regra, 'cliente_id' AS coluna, total AS total_linhas, nulos_id AS linhas_invalidas, CASE WHEN nulos_id > 0 THEN 'FALHA' ELSE 'OK' END AS status FROM c_cli
UNION ALL
SELECT 'silver.clientes', 'unique', 'cliente_id', total, dup_id, CASE WHEN dup_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_cli
UNION ALL
SELECT 'silver.clientes', 'not_null', 'email', total, nulos_email, CASE WHEN nulos_email > 0 THEN 'AVISO' ELSE 'OK' END FROM c_cli
UNION ALL
SELECT 'silver.pedidos', 'not_null', 'pedido_id', total, nulos_id, CASE WHEN nulos_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_ped
UNION ALL
SELECT 'silver.pedidos', 'unique', 'pedido_id', total, dup_id, CASE WHEN dup_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_ped
UNION ALL
SELECT 'silver.pedidos', 'in_set', 'status', total, inv_status, CASE WHEN inv_status > 0 THEN 'FALHA' ELSE 'OK' END FROM c_ped
UNION ALL
SELECT 'silver.avaliacoes', 'unique', 'avaliacao_id', total, dup_id, CASE WHEN dup_id > 0 THEN 'FALHA' ELSE 'OK' END FROM c_av
UNION ALL
SELECT 'silver.avaliacoes', 'between', 'nota', total, inv_nota, CASE WHEN inv_nota > 0 THEN 'FALHA' ELSE 'OK' END FROM c_av
""")

# Grava diretamente no Iceberg
rel.writeTo("lake.silver.dq_resultados").using("iceberg").createOrReplace()

# Exibe o resultado lendo a tabela direto do catálogo
spark.table("lake.silver.dq_resultados").show(truncate=False)

+-----------------+--------+------------+------------+----------------+------+
|tabela           |regra   |coluna      |total_linhas|linhas_invalidas|status|
+-----------------+--------+------------+------------+----------------+------+
|silver.clientes  |not_null|cliente_id  |5000        |0               |OK    |
|silver.clientes  |unique  |cliente_id  |5000        |0               |OK    |
|silver.clientes  |not_null|email       |5000        |270             |AVISO |
|silver.pedidos   |not_null|pedido_id   |29960       |0               |OK    |
|silver.pedidos   |unique  |pedido_id   |29960       |0               |OK    |
|silver.pedidos   |in_set  |status      |29960       |0               |OK    |
|silver.avaliacoes|unique  |avaliacao_id|6464        |0               |OK    |
|silver.avaliacoes|between |nota        |6464        |0               |OK    |
+-----------------+--------+------------+------------+----------------+------+



## Histórico de versões: a Silver tem memória
Cada escrita no Iceberg cria um **snapshot**: uma versão completa da tabela naquele instante. Nada é editado no lugar. Isso responde a perguntas de governança:
**o que mudou, quando, e dá para voltar?**

In [10]:
spark.sql("""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records']   AS linhas_adicionadas,
           summary['total-records']   AS total_de_linhas
    FROM lake.silver.itens_pedido_aula.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-----------------+---------+------------------+---------------+
|committed_at           |snapshot_id      |operation|linhas_adicionadas|total_de_linhas|
+-----------------------+-----------------+---------+------------------+---------------+
|2026-10-24 21:43:02.924|12986276392828925|append   |50105             |50105          |
+-----------------------+-----------------+---------+------------------+---------------+



Vamos simular um erro humano: alguém "limpa" os pedidos cancelados achando que eram lixo.
Guardamos o id da versão **antes** do estrago.

In [11]:
TABELA = "lake.silver.itens_pedido_aula"

snap_bom = spark.sql(f"SELECT snapshot_id FROM {TABELA}.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0]
antes = spark.table(TABELA).count()

spark.sql(f"DELETE FROM {TABELA} WHERE status = 'cancelado'")        # a regra errada

depois = spark.table(TABELA).count()
print(f"linhas: {antes} -> {depois}  (sumiram {antes - depois})")
print("snapshot bom:", snap_bom)

linhas: 50105 -> 43886  (sumiram 6219)
snapshot bom: 12986276392828925


In [12]:
# voltar no tempo (só ler)
spark.sql(f"SELECT count(*) AS linhas_na_versao_antiga FROM {TABELA} VERSION AS OF {snap_bom}").show()
spark.sql(f"SELECT count(*) AS linhas_na_versao_atual   FROM {TABELA}").show()

+-----------------------+
|linhas_na_versao_antiga|
+-----------------------+
|                  50105|
+-----------------------+

+----------------------+
|linhas_na_versao_atual|
+----------------------+
|                 43886|
+----------------------+



In [13]:

spark.sql(f"CALL lake.system.rollback_to_snapshot('silver.itens_pedido_aula', {snap_bom})")
print("linhas depois do rollback:", spark.table(TABELA).count())

linhas depois do rollback: 50105


Se alguma regra aparece com status `AVISO` ou `ERRO`, ótimo: é o ponto de conversa. (Dica: e-mail inválido vira `NULL` na Silver, e a regra `not_null(email)` aparece como `AVISO`: o cliente continua existindo, só perdeu o contato.)

## Checkpoint
- [ ] Silver com 5 tabelas + `quarentena` + `dq_resultados`.
- [ ] Você consegue explicar por que um pedido foi para a quarentena.

➡️ Próximo: `03_gold.ipynb` (o foco da aula)